In [3]:

import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# 1. Calea catre dataset
cale = "/kaggle/input/datasets/nunenuh/pytorch-challange-flower-dataset/dataset"

# 2. Preprocesarea imaginilor
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 3. Clasa Dataset
class FlowerDataset(Dataset):
    def __init__(self, cale, transform=None):
        self.imagini = []
        self.etichete = []
        self.transform = transform

        clase = sorted(os.listdir(cale))

        for eticheta, clasa in enumerate(clase):
            folder = os.path.join(cale, clasa)

            if os.path.isdir(folder):
                for fisier in os.listdir(folder):
                    if fisier.lower().endswith((".jpg", ".jpeg", ".png")):
                        self.imagini.append(os.path.join(folder, fisier))
                        self.etichete.append(eticheta)

    def __len__(self):
        return len(self.imagini)

    def __getitem__(self, index):
        imagine = Image.open(self.imagini[index]).convert("RGB")
        eticheta = self.etichete[index]

        if self.transform:
            imagine = self.transform(imagine)

        return imagine, eticheta


# 4. Incarcarea datelor
dataset = FlowerDataset(cale)

total = len(dataset)

# 5. Impartirea in train / validation / test
n_train = int(0.70 * total)
n_val = int(0.15 * total)
n_test = total - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# Aplicam transformari diferite pentru train si test
train_set.dataset = FlowerDataset(cale, transform_train)
val_set.dataset = FlowerDataset(cale, transform_test)
test_set.dataset = FlowerDataset(cale, transform_test)

# 6. Crearea DataLoader-elor
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

# 7. Verificarea functionarii
print("Total imagini:", total)
print("Train:", len(train_set))
print("Validation:", len(val_set))
print("Test:", len(test_set))

imagini, etichete = next(iter(train_loader))

print("Dimensiune imagini:", imagini.shape)
print("Dimensiune etichete:", etichete.shape)
print("Numar clase:", len(set(dataset.etichete)))


Total imagini: 819
Train: 573
Validation: 122
Test: 124
Dimensiune imagini: torch.Size([32, 3, 224, 224])
Dimensiune etichete: torch.Size([32])
Numar clase: 1
